In [1]:
import sqlite3

import pandas as pd
from starter import collector, rag

#### Q1. First trace

In [2]:
query = "How does the agentic loop keep calling the model until it stops?"
rag.rag(query)

print("Number of spans in the trace: ", len(collector.spans))

Number of spans in the trace:  3


#### Q2. Capturing metrics as span attributes

In [3]:
spans = {s.name: s for s in collector.spans}
llm_span = spans["llm"]
input_tokens = llm_span.attributes["input_tokens"]
print("Input tokens:", input_tokens)

Input tokens: 6954.0


#### Q3. Span timing

In [4]:
llm_duration = (llm_span.end_time - llm_span.start_time) / 1_000_000_000
print(f"LLM span duration: {llm_duration} seconds")

LLM span duration: 3.406307162 seconds


#### Q4. Saving traces to SQLite

In [5]:
with sqlite3.connect("traces.db") as conn:
    cursor = conn.execute("SELECT DISTINCT name FROM spans")
    span_names = [row[0] for row in cursor.fetchall()]

print("Span names in database:", span_names)

Span names in database: ['search', 'llm', 'rag']


#### Q5. Querying trace data

In [6]:
with sqlite3.connect("traces.db") as conn:
    cursor = conn.execute("""
        SELECT name, SUM(end_time - start_time) as total_duration_ns
        FROM spans 
        WHERE name != 'rag'
        GROUP BY name
        ORDER BY total_duration_ns DESC
    """)
    span_durations = {}
    for row in cursor.fetchall():
        span_name, duration_ns = row
        duration_seconds = duration_ns / 1_000_000_000  # Convert nanoseconds to seconds
        span_durations[span_name] = duration_seconds
        print(f"    {span_name}: {duration_seconds:.4f} seconds")

    llm: 3.4063 seconds
    search: 0.0039 seconds


#### Q6. Token stability across runs

In [7]:
for _ in range(3):
    rag.rag(query)

with sqlite3.connect("traces.db") as conn:
    df = pd.read_sql_query("SELECT input_tokens FROM spans WHERE name = 'llm' ORDER BY start_time", conn)

print("Input tokens across all llm spans in database:", df)

Input tokens across all llm spans in database:    input_tokens
0          6954
1          6954
2          6954
3          6954
